    # Week 1: Linear algebra for attention

    [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuranLearning/ai-llm-deep-dive/blob/main/notebooks/week-01-linear-algebra.ipynb)
    · [Lesson text](https://github.com/QuranLearning/ai-llm-deep-dive/blob/main/lessons/week-01.md) · [Learning Hub week page](https://claude.ai/artifact/VwTfCas4FMnB23QHuBUJhj#w-ai-llm-1)

    **By the end of this notebook you will:**
    - compute dot products, cosine similarity and matrix products by hand and with NumPy
- write a numerically stable softmax
- build a single self-attention head, with and without a causal mask

    **How to use it:** work top to bottom. Each exercise has a starter cell (replace the `TODO`s) and a **Check** cell that tells you if you got it right.
    Stuck for more than 15 minutes? Ask the tutor on the week page, then peek at the **Solutions** section at the bottom.
    In Colab, use *File → Save a copy in Drive* so your work is saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)
def ok(msg="Correct"): print("\u2705", msg)

## Exercise 1: dot product and cosine similarity
Write `dot(a, b)` with a plain Python loop (no `np.dot`), and `cosine(a, b)` = a·b / (|a| |b|).
The lesson's example: q = [1, 2], k1 = [2, 1], k2 = [-1, 0.5] gives q·k1 = 4 and q·k2 = 0.

In [ ]:
def dot(a, b):
    # TODO: loop over the elements and accumulate a[i] * b[i]
    raise NotImplementedError

def cosine(a, b):
    # TODO: use dot() and np.sqrt
    raise NotImplementedError

In [ ]:
q, k1, k2 = [1, 2], [2, 1], [-1, 0.5]
assert dot(q, k1) == 4 and dot(q, k2) == 0
a, b = rng.normal(size=8), rng.normal(size=8)
assert np.isclose(dot(a, b), a @ b)
assert np.isclose(cosine([1, 0], [0, 1]), 0) and np.isclose(cosine([2, 2], [1, 1]), 1)
ok("dot and cosine")

## Exercise 2: matrix multiplication = a batch of dot products
Write `matmul(A, B)` with loops: `C[i, j]` is row i of A dotted with column j of B. Raise a `ValueError` when the inner sizes don't match.

In [ ]:
def matmul(A, B):
    A, B = np.asarray(A, float), np.asarray(B, float)
    # TODO: check shapes, create C = np.zeros((m, p)), fill it with two loops and dot()
    raise NotImplementedError

In [ ]:
A, B = rng.normal(size=(3, 4)), rng.normal(size=(4, 2))
assert np.allclose(matmul(A, B), A @ B)
try:
    matmul(A, A); raise AssertionError("expected a ValueError for (3,4) @ (3,4)")
except ValueError:
    pass
ok("matmul")

## Exercise 3: shapes without running code
Fill in the shapes as tuples. Inner sizes must match; outer sizes survive.

In [ ]:
# x: (32, 768) batch of token vectors, W: (768, 3072), Q and K: (10, 64)
shape_xW   = None   # TODO: shape of x @ W
shape_QKt  = None   # TODO: shape of Q @ K.T
shape_Wt_x = None   # TODO: shape of W.T @ x.T

In [ ]:
assert shape_xW == (32, 3072), "x @ W"
assert shape_QKt == (10, 10), "Q @ K.T"
assert shape_Wt_x == (3072, 32), "W.T @ x.T = (x @ W).T"
ok("shapes")

## Exercise 4: a stable softmax
`softmax(z)` along the last axis. Subtract the max first so `exp` never overflows. It must work for a vector and row-wise for a matrix.

In [ ]:
def softmax(z):
    z = np.asarray(z, float)
    # TODO: subtract z.max(axis=-1, keepdims=True), exponentiate, normalise along the last axis
    raise NotImplementedError

In [ ]:
assert np.allclose(softmax([4, 0]), [0.982, 0.018], atol=1e-3)
assert np.allclose(softmax([1000, 1000]), [0.5, 0.5])          # no overflow
M = rng.normal(size=(5, 7))
assert np.allclose(softmax(M).sum(axis=1), 1)
print("softmax([2, 2, 0]) =", softmax([2, 2, 0]))
ok("softmax")

## Exercise 5: one self-attention head
Implement the lesson's formula. `X` is (T, d); the weight matrices are (d, d_k).

```
Q, K, V = X Wq, X Wk, X Wv
scores  = Q Kᵀ / sqrt(d_k)
weights = softmax(scores)          # row-wise
out     = weights V
```
With `causal=True`, set `scores[i, j] = -inf` for every `j > i` before the softmax, so a token only sees itself and earlier tokens. Return `(out, weights)`.

In [ ]:
def attention(X, Wq, Wk, Wv, causal=False):
    # TODO
    raise NotImplementedError

In [ ]:
T, d = 5, 8
X = rng.normal(size=(T, d)); Wq, Wk, Wv = (rng.normal(size=(d, d)) for _ in range(3))
out, W = attention(X, Wq, Wk, Wv)
assert out.shape == (T, d) and W.shape == (T, T)
assert np.allclose(W.sum(axis=1), 1)
out_c, Wc = attention(X, Wq, Wk, Wv, causal=True)
assert np.allclose(np.triu(Wc, k=1), 0), "future positions must get zero weight"
assert np.isclose(Wc[0, 0], 1), "the first token can only attend to itself"
assert np.allclose(out_c[0], (X @ Wv)[0]), "so its output is its own value vector"
ok("attention")

## Explore (no check)
Plot the two weight matrices. Then change one row of `X` and re-plot: which rows and columns move? Write two sentences about it in your weekly journal.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
for a_, M_, t in zip(ax, [W, Wc], ["full attention", "causal attention"]):
    im = a_.imshow(M_, vmin=0, vmax=1, cmap="viridis"); a_.set_title(t); a_.set_xlabel("key j"); a_.set_ylabel("query i")
fig.colorbar(im, ax=ax, shrink=0.8); plt.show()

---
## Solutions

Try every exercise first. Run the cell below only to compare with your own version; it overwrites your functions, so re-run the Check cells afterwards.

In [ ]:
def dot(a, b):
    s = 0.0
    for x, y in zip(a, b):
        s += x * y
    return s

def cosine(a, b):
    return dot(a, b) / (np.sqrt(dot(a, a)) * np.sqrt(dot(b, b)))

def matmul(A, B):
    A, B = np.asarray(A, float), np.asarray(B, float)
    (m, n), (n2, p) = A.shape, B.shape
    if n != n2:
        raise ValueError(f"inner sizes differ: {A.shape} @ {B.shape}")
    C = np.zeros((m, p))
    for i in range(m):
        for j in range(p):
            C[i, j] = dot(A[i], B[:, j])
    return C

shape_xW, shape_QKt, shape_Wt_x = (32, 3072), (10, 10), (3072, 32)

def softmax(z):
    z = np.asarray(z, float)
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def attention(X, Wq, Wk, Wv, causal=False):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    S = Q @ K.T / np.sqrt(K.shape[1])
    if causal:
        S = np.where(np.triu(np.ones_like(S, dtype=bool), k=1), -np.inf, S)
    Wt = softmax(S)
    return Wt @ V, Wt